In [1]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================

!pip install -q -U \
    langgraph \
    langchain \
    langchain-google-genai \
    pydantic \
    pyyaml \
    tavily-python \
    ddgs \
    tabulate


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 16.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curren

In [2]:
# ============================================================
# CELL 2 — IMPORTS & CONFIGURATION
# ============================================================

import os
import time
import operator

from datetime import datetime
from getpass import getpass
from typing import Annotated, TypedDict

from pydantic import BaseModel, Field

from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, END
from langgraph.types import Send


# ------------------------------------------------------------
# API KEY
# ------------------------------------------------------------

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass(
        "Enter your Google Gemini API key: "
    )

print("API key configured.")


# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

CURRENT_YEAR = datetime.now().year

SEARCH_CONFIG = {
    "quick": {
        "general_results": 4,
        "financial_results": 4,
        "fact_check_claims": 4,
        "context_sources": 10,
    },
    "standard": {
        "general_results": 6,
        "financial_results": 6,
        "fact_check_claims": 8,
        "context_sources": 15,
    },
    "deep": {
        "general_results": 10,
        "financial_results": 10,
        "fact_check_claims": 12,
        "context_sources": 20,
    },
}

DEFAULT_ANALYSIS_DEPTH = "standard"

print(f"Current year: {CURRENT_YEAR}")
print(f"Default analysis depth: {DEFAULT_ANALYSIS_DEPTH}")


Enter your Google Gemini API key: ··········
API key configured.
Current year: 2026
Default analysis depth: standard


In [3]:
# ============================================================
# CELL 3 — SHARED LANGGRAPH STATE
# ============================================================

class AgentFinding(TypedDict, total=False):
    """
    Standardized finding produced by research agents.
    """
    agent: str
    category: str
    title: str
    detail: str
    severity: str
    confidence: float
    sources: list[str]
    verified: bool


class DueDiligenceState(TypedDict, total=False):
    """
    Shared state passed through the LangGraph pipeline.
    """

    # --------------------------------------------------------
    # Input
    # --------------------------------------------------------
    company_name: str
    query: str
    analysis_depth: str

    # --------------------------------------------------------
    # Planning
    # --------------------------------------------------------
    research_plan: dict
    focus_areas: list[str]

    # --------------------------------------------------------
    # Specialist findings
    # --------------------------------------------------------
    financial_findings: Annotated[
        list[AgentFinding],
        operator.add
    ]

    news_findings: Annotated[
        list[AgentFinding],
        operator.add
    ]

    competitive_findings: Annotated[
        list[AgentFinding],
        operator.add
    ]

    risk_findings: Annotated[
        list[AgentFinding],
        operator.add
    ]

    # --------------------------------------------------------
    # Verification
    # --------------------------------------------------------
    fact_check_results: Annotated[
        list[dict],
        operator.add
    ]

    contradictions: Annotated[
        list[dict],
        operator.add
    ]

    # --------------------------------------------------------
    # Final output
    # --------------------------------------------------------
    executive_summary: str
    final_report: str
    overall_risk_rating: str
    overall_confidence: float

    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------
    pipeline_trace: Annotated[
        list[dict],
        operator.add
    ]

    errors: Annotated[
        list[str],
        operator.add
    ]

    status: str


print("State schema created.")
print(f"Fields: {len(DueDiligenceState.__annotations__)}")


State schema created.
Fields: 18


In [4]:
# ============================================================
# CELL 4 — STRUCTURED OUTPUT SCHEMAS
# ============================================================


class ResearchTask(BaseModel):
    """
    A single task generated by the Lead Analyst.
    """

    agent: str = Field(
        description=(
            "Target specialist: financial, news, "
            "competitive, or risk"
        )
    )

    task: str = Field(
        description="Specific research task"
    )

    priority: str = Field(
        description="high, medium, or low"
    )


class ResearchPlan(BaseModel):
    """
    Lead Analyst research plan.
    """

    company_summary: str

    sub_tasks: list[ResearchTask] = Field(
        default_factory=list
    )

    focus_areas: list[str] = Field(
        default_factory=list
    )

    risk_hypothesis: str = ""


class FinancialAnalysis(BaseModel):
    """
    Financial Analyst structured output.
    """

    company_name: str

    financial_health_rating: str = Field(
        description=(
            "strong, moderate, weak, critical, "
            "or insufficient_data"
        )
    )

    revenue_analysis: str = ""

    profitability_analysis: str = ""

    red_flags: list[str] = Field(
        default_factory=list
    )

    green_flags: list[str] = Field(
        default_factory=list
    )

    data_gaps: list[str] = Field(
        default_factory=list
    )

    sources: list[str] = Field(
        default_factory=list
    )


class NewsEvent(BaseModel):
    """
    One structured news event.
    """

    date: str = ""

    headline: str

    sentiment: str = Field(
        description="positive, negative, or neutral"
    )

    impact: str = Field(
        description="high, medium, or low"
    )

    summary: str

    source: str = ""


class NewsAnalysis(BaseModel):
    """
    News Analyst structured output.
    """

    company_name: str

    overall_sentiment: str = Field(
        description="positive, negative, neutral, or mixed"
    )

    events: list[NewsEvent] = Field(
        default_factory=list
    )

    overall_summary: str = ""

    sources: list[str] = Field(
        default_factory=list
    )


class Competitor(BaseModel):
    """
    Competitor information.
    """

    name: str

    comparison: str

    threat_level: str = Field(
        description="high, medium, or low"
    )


class CompetitiveAnalysis(BaseModel):
    """
    Competitive Intelligence structured output.
    """

    company_name: str

    market_position: str = ""

    competitive_advantages: list[str] = Field(
        default_factory=list
    )

    competitive_threats: list[str] = Field(
        default_factory=list
    )

    competitors: list[Competitor] = Field(
        default_factory=list
    )

    sources: list[str] = Field(
        default_factory=list
    )


class RiskItem(BaseModel):
    """
    Individual risk.
    """

    title: str

    category: str

    description: str

    severity: str = Field(
        description="critical, high, medium, low"
    )

    likelihood: str = Field(
        description="high, medium, or low"
    )

    source: str = ""


class RiskAssessment(BaseModel):
    """
    Risk Assessor structured output.
    """

    company_name: str

    overall_risk_level: str = Field(
        description="critical, high, moderate, or low"
    )

    risks: list[RiskItem] = Field(
        default_factory=list
    )

    risk_summary: str = ""

    sources: list[str] = Field(
        default_factory=list
    )


class Verification(BaseModel):
    """
    Verification of one claim.
    """

    claim: str

    agent: str

    status: str = Field(
        description=(
            "verified, contradicted, or unverifiable"
        )
    )

    reasoning: str

    supporting_sources: list[str] = Field(
        default_factory=list
    )


class FactCheckReport(BaseModel):
    """
    Fact-checking result.
    """

    total_claims_checked: int

    verified_count: int

    contradicted_count: int

    unverifiable_count: int

    verifications: list[Verification] = Field(
        default_factory=list
    )

    cross_agent_contradictions: list[str] = Field(
        default_factory=list
    )

    overall_reliability: str = Field(
        default="moderate",
        description="high, moderate, or low"
    )


class ExecutiveSummary(BaseModel):
    """
    Final report synthesis.
    """

    company_name: str

    one_line_verdict: str

    overall_risk_rating: str

    overall_confidence: float = Field(
        ge=0.0,
        le=1.0
    )

    key_strengths: list[str] = Field(
        default_factory=list
    )

    key_risks: list[str] = Field(
        default_factory=list
    )

    recommendation: str

    action_items: list[str] = Field(
        default_factory=list
    )


print("Structured schemas created.")


Structured schemas created.


In [6]:
# ============================================================
# CELL 5 — GEMINI LLM
# ============================================================

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key="",
    temperature=0.1,
    max_output_tokens=4096,
)

# Basic test
response = llm.invoke(
    "What is 2+2? Reply with only the number."
)

print("LLM test:", response.content)
print("Gemini connection successful.")


LLM test: [{'type': 'text', 'text': '4', 'extras': {'signature': 'EusCCugCARFNMg92OD5M15rJAXpuvS3qArvyr/14A7wvkBCXDV3R5PE44NzOIOSgTbLvUgVVYrZrBFeiVMK2/5bCZCKrUKEioDfutIMGVExS2DUKBciIfnkWoY7QKDrLKcqynnicYLbI8DW+OSC/y+xYqCs4B50mCG02Bjg/a999TExe92fhAyMUamQmUVmdvYfMITwzexFvOfEJhZneZN0cMLXLRj1B3hYfSiJxhq6sAefItAivTj2/e4LBFpUj7FK/5i6GqZHP3lqejHwZA924xOHYdi4U4Xc+s49H8CqAiIc9aoQA1WpVlaebjumc7t/ymV04PiO+6XmASNZq/OROmIAD9/Z6HbREZ5FDEGUiqQXegm3d2xs6f3rMvBzyImrZyD1EacApH4Tm7tgEZ0Fy/OkmXpYgl4HUq3Z6fiwcJFFJM47WMtAdQvCGgkW+/qEOYSYWTUgakE/PPKZVcwH4Rn3b3YH2gTIiWp6I'}}]
Gemini connection successful.


In [7]:
# ============================================================
# CELL 6 — WEB SEARCH
# ============================================================

try:
    from ddgs import DDGS
except ImportError:
    from duckduckgo_search import DDGS


def web_search(
    query: str,
    max_results: int = 5
) -> list[dict]:
    """
    Search the web using DuckDuckGo.

    Returns normalized search results.
    """

    try:
        with DDGS() as ddgs:
            results = list(
                ddgs.text(
                    query,
                    max_results=max_results
                )
            )

        return [
            {
                "title": r.get("title", ""),
                "url": r.get("href", ""),
                "snippet": r.get("body", "")[:700],
            }
            for r in results
        ]

    except Exception as e:
        print(f"Search failed for '{query}': {e}")
        return []


def build_search_context(
    results: list[dict],
    max_sources: int = 15
) -> str:
    """
    Convert search results into LLM-readable context.
    """

    lines = []
    seen = set()

    for result in results:

        url = result.get("url", "")

        if not url or url in seen:
            continue

        seen.add(url)

        lines.append(
            f"Source URL: {url}\n"
            f"Title: {result.get('title', '')}\n"
            f"Content: {result.get('snippet', '')}\n"
        )

    return "\n---\n".join(
        lines[:max_sources]
    )


print("Search functions ready.")


Search functions ready.


In [8]:
# ============================================================
# CELL 7 — HELPERS
# ============================================================


def get_depth_config(state: DueDiligenceState) -> dict:
    """
    Return search configuration based on analysis depth.
    """

    depth = state.get(
        "analysis_depth",
        DEFAULT_ANALYSIS_DEPTH
    )

    return SEARCH_CONFIG.get(
        depth,
        SEARCH_CONFIG[DEFAULT_ANALYSIS_DEPTH]
    )


def get_agent_tasks(
    state: DueDiligenceState,
    agent_name: str
) -> list[str]:
    """
    Extract planner-generated tasks for a specialist.
    """

    plan = state.get("research_plan", {})

    tasks = plan.get("sub_tasks", [])

    selected = []

    for task in tasks:

        if isinstance(task, dict):

            if task.get("agent") == agent_name:
                selected.append(
                    task.get("task", "")
                )

        elif isinstance(task, ResearchTask):

            if task.agent == agent_name:
                selected.append(task.task)

    return selected


def get_planning_context(
    state: DueDiligenceState,
    agent_name: str
) -> str:
    """
    Build planning context for specialist agents.
    """

    plan = state.get("research_plan", {})

    tasks = get_agent_tasks(
        state,
        agent_name
    )

    focus_areas = state.get(
        "focus_areas",
        []
    )

    risk_hypothesis = plan.get(
        "risk_hypothesis",
        ""
    )

    return f"""
RESEARCH PLAN

Company summary:
{plan.get("summary", "")}

Assigned tasks:
{chr(10).join("- " + t for t in tasks)}

Focus areas:
{", ".join(focus_areas)}

Initial risk hypothesis:
{risk_hypothesis}
"""


def safe_sources(results: list[dict]) -> list[str]:
    """
    Extract valid source URLs.
    """

    return [
        r["url"]
        for r in results
        if r.get("url")
    ]


In [9]:
# ============================================================
# CELL 8 — LEAD ANALYST: PLANNER
# ============================================================


def plan_research(
    state: DueDiligenceState
) -> dict:

    company = state["company_name"]
    query = state.get(
        "query",
        f"Comprehensive due diligence of {company}"
    )

    start = time.time()

    print(
        f"\n[Lead Analyst] "
        f"Planning research for {company}"
    )

    try:

        structured = llm.with_structured_output(
            ResearchPlan
        )

        plan = structured.invoke(
            f"""
You are the Lead Analyst responsible for planning
a company due-diligence investigation.

Company:
{company}

User research objective:
{query}

Create a practical research plan for four specialist
agents:

1. financial
2. news
3. competitive
4. risk

Generate specific, evidence-oriented tasks.

Do not invent company facts.
The tasks should guide web research.

Identify:
- company context
- specialist research tasks
- key focus areas
- an initial risk hypothesis

Return only the requested structured output.
"""
        )

        research_plan = {
            "summary": plan.company_summary,
            "sub_tasks": [
                task.model_dump()
                for task in plan.sub_tasks
            ],
            "risk_hypothesis": plan.risk_hypothesis,
        }

        duration = round(
            time.time() - start,
            2
        )

        print(
            f"  Created {len(plan.sub_tasks)} tasks "
            f"in {duration}s"
        )

        return {
            "research_plan": research_plan,
            "focus_areas": plan.focus_areas,
            "status": "researching",
            "pipeline_trace": [
                {
                    "agent": "lead_planner",
                    "duration": duration,
                }
            ],
        }

    except Exception as e:

        print(
            f"  [Planner] FALLBACK: {e}"
        )

        return {
            "research_plan": {
                "summary": "",
                "sub_tasks": [],
                "risk_hypothesis": "",
            },
            "focus_areas": [
                "financial",
                "news",
                "competitive",
                "risk",
            ],
            "status": "researching",
            "errors": [str(e)],
            "pipeline_trace": [
                {
                    "agent": "lead_planner",
                    "duration": round(
                        time.time() - start,
                        2
                    ),
                }
            ],
        }


print("Planner ready.")


Planner ready.


In [10]:
# ============================================================
# CELL 9 — FINANCIAL ANALYST
# ============================================================


def financial_analyst(
    state: DueDiligenceState
) -> dict:

    company = state["company_name"]
    config = get_depth_config(state)

    start = time.time()

    print(
        f"  [Financial] Researching {company}..."
    )

    try:

        tasks = get_agent_tasks(
            state,
            "financial"
        )

        queries = [
            f"{company} latest annual report financial results revenue profit",
            f"{company} latest balance sheet cash debt profitability",
            f"{company} funding valuation financial performance",
        ]

        if tasks:
            queries.append(
                f"{company} " + " ".join(tasks[:2])
            )

        results = []

        for query in queries:
            results.extend(
                web_search(
                    query,
                    config["financial_results"]
                )
            )

        context = build_search_context(
            results,
            config["context_sources"]
        )

        planning_context = get_planning_context(
            state,
            "financial"
        )

        structured = llm.with_structured_output(
            FinancialAnalysis
        )

        analysis = structured.invoke(
            f"""
You are a Financial Analyst conducting company
due diligence.

Company:
{company}

{planning_context}

Use ONLY information supported by the supplied
search results.

Do not invent financial numbers.

Evaluate:
- revenue
- growth
- profitability
- balance sheet
- funding/valuation where relevant
- financial red flags
- financial strengths
- missing information

Search results:

{context}
"""
        )

        findings = []

        for flag in analysis.red_flags:

            findings.append({
                "agent": "financial",
                "category": "financial",
                "title": f"Financial concern: {flag}",
                "detail": flag,
                "severity": "high",
                "confidence": 0.70,
                "sources": analysis.sources,
                "verified": False,
            })

        for flag in analysis.green_flags:

            findings.append({
                "agent": "financial",
                "category": "financial",
                "title": f"Financial strength: {flag}",
                "detail": flag,
                "severity": "info",
                "confidence": 0.70,
                "sources": analysis.sources,
                "verified": False,
            })

        findings.append({
            "agent": "financial",
            "category": "financial",
            "title": (
                f"Financial health: "
                f"{analysis.financial_health_rating}"
            ),
            "detail": (
                f"Revenue analysis: "
                f"{analysis.revenue_analysis}\n\n"
                f"Profitability analysis: "
                f"{analysis.profitability_analysis}"
            ),
            "severity": "info",
            "confidence": 0.75,
            "sources": analysis.sources,
            "verified": False,
        })

        if analysis.data_gaps:

            findings.append({
                "agent": "financial",
                "category": "financial",
                "title": "Financial data gaps",
                "detail": "; ".join(
                    analysis.data_gaps
                ),
                "severity": "medium",
                "confidence": 0.65,
                "sources": analysis.sources,
                "verified": False,
            })

        duration = round(
            time.time() - start,
            2
        )

        print(
            f"  [Financial] "
            f"{analysis.financial_health_rating} | "
            f"{len(findings)} findings | "
            f"{duration}s"
        )

        return {
            "financial_findings": findings,
            "pipeline_trace": [
                {
                    "agent": "financial",
                    "duration": duration,
                }
            ],
        }

    except Exception as e:

        print(
            f"  [Financial] FALLBACK: {e}"
        )

        return {
            "financial_findings": [
                {
                    "agent": "financial",
                    "category": "financial",
                    "title": "Financial analysis failed",
                    "detail": str(e),
                    "severity": "low",
                    "confidence": 0.0,
                    "sources": [],
                    "verified": False,
                }
            ],
            "errors": [str(e)],
        }


print("Financial Analyst ready.")


Financial Analyst ready.


In [11]:
# ============================================================
# CELL 10 — NEWS & SENTIMENT ANALYST
# ============================================================


def news_sentiment(
    state: DueDiligenceState
) -> dict:

    company = state["company_name"]
    config = get_depth_config(state)

    start = time.time()

    print(
        f"  [News] Researching {company}..."
    )

    try:

        tasks = get_agent_tasks(
            state,
            "news"
        )

        queries = [
            f"{company} latest news",
            f"{company} recent controversy criticism",
            f"{company} recent positive developments",
        ]

        if tasks:
            queries.append(
                f"{company} " + " ".join(tasks[:2])
            )

        results = []

        for query in queries:

            results.extend(
                web_search(
                    query,
                    config["general_results"]
                )
            )

        context = build_search_context(
            results,
            config["context_sources"]
        )

        planning_context = get_planning_context(
            state,
            "news"
        )

        structured = llm.with_structured_output(
            NewsAnalysis
        )

        analysis = structured.invoke(
            f"""
You are a News and Sentiment Analyst.

Company:
{company}

{planning_context}

Analyze recent company-related news using ONLY
the supplied search results.

For each significant event provide:
- date if available
- headline
- sentiment
- impact
- concise summary
- source URL when available

Avoid unsupported claims.

Search results:

{context}
"""
        )

        findings = []

        for event in analysis.events:

            severity_map = {
                "high": "high",
                "medium": "medium",
                "low": "low",
            }

            findings.append({
                "agent": "news",
                "category": "news",
                "title": event.headline,
                "detail": (
                    f"Date: {event.date}\n"
                    f"Sentiment: {event.sentiment}\n"
                    f"Impact: {event.impact}\n"
                    f"Summary: {event.summary}"
                ),
                "severity": severity_map.get(
                    event.impact.lower(),
                    "medium"
                ),
                "confidence": 0.70,
                "sources": (
                    [event.source]
                    if event.source
                    else []
                ),
                "verified": False,
            })

        findings.append({
            "agent": "news",
            "category": "news",
            "title": (
                f"Overall news sentiment: "
                f"{analysis.overall_sentiment}"
            ),
            "detail": analysis.overall_summary,
            "severity": "info",
            "confidence": 0.65,
            "sources": (
                analysis.sources
                or safe_sources(results[:5])
            ),
            "verified": False,
        })

        duration = round(
            time.time() - start,
            2
        )

        print(
            f"  [News] "
            f"{len(analysis.events)} events | "
            f"{duration}s"
        )

        return {
            "news_findings": findings,
            "pipeline_trace": [
                {
                    "agent": "news",
                    "duration": duration,
                }
            ],
        }

    except Exception as e:

        print(
            f"  [News] FALLBACK: {e}"
        )

        return {
            "news_findings": [
                {
                    "agent": "news",
                    "category": "news",
                    "title": "News analysis failed",
                    "detail": str(e),
                    "severity": "low",
                    "confidence": 0.0,
                    "sources": [],
                    "verified": False,
                }
            ],
            "errors": [str(e)],
        }


print("News Analyst ready.")


News Analyst ready.


In [12]:
# ============================================================
# CELL 11 — COMPETITIVE INTELLIGENCE
# ============================================================


def competitive_intel(
    state: DueDiligenceState
) -> dict:

    company = state["company_name"]
    config = get_depth_config(state)

    start = time.time()

    print(
        f"  [Competitive] Researching {company}..."
    )

    try:

        tasks = get_agent_tasks(
            state,
            "competitive"
        )

        queries = [
            f"{company} competitors market share industry",
            f"{company} competitive advantages disadvantages",
            f"{company} industry market position comparison",
        ]

        if tasks:
            queries.append(
                f"{company} " + " ".join(tasks[:2])
            )

        results = []

        for query in queries:

            results.extend(
                web_search(
                    query,
                    config["general_results"]
                )
            )

        context = build_search_context(
            results,
            config["context_sources"]
        )

        planning_context = get_planning_context(
            state,
            "competitive"
        )

        structured = llm.with_structured_output(
            CompetitiveAnalysis
        )

        analysis = structured.invoke(
            f"""
You are a Competitive Intelligence Analyst.

Company:
{company}

{planning_context}

Analyze the company's competitive position using
ONLY evidence in the search results.

Identify:
- market position
- important competitors
- competitive advantages
- competitive threats
- relative threat level of competitors

Do not invent market-share numbers.

Search results:

{context}
"""
        )

        findings = []

        findings.append({
            "agent": "competitive",
            "category": "competitive",
            "title": "Market position",
            "detail": analysis.market_position,
            "severity": "info",
            "confidence": 0.65,
            "sources": (
                analysis.sources
                or safe_sources(results[:5])
            ),
            "verified": False,
        })

        for advantage in analysis.competitive_advantages:

            findings.append({
                "agent": "competitive",
                "category": "competitive",
                "title": (
                    f"Competitive advantage: "
                    f"{advantage}"
                ),
                "detail": advantage,
                "severity": "info",
                "confidence": 0.65,
                "sources": analysis.sources,
                "verified": False,
            })

        for threat in analysis.competitive_threats:

            findings.append({
                "agent": "competitive",
                "category": "competitive",
                "title": (
                    f"Competitive threat: "
                    f"{threat}"
                ),
                "detail": threat,
                "severity": "medium",
                "confidence": 0.65,
                "sources": analysis.sources,
                "verified": False,
            })

        for competitor in analysis.competitors:

            severity_map = {
                "high": "high",
                "medium": "medium",
                "low": "low",
            }

            findings.append({
                "agent": "competitive",
                "category": "competitive",
                "title": (
                    f"Competitor: "
                    f"{competitor.name}"
                ),
                "detail": competitor.comparison,
                "severity": severity_map.get(
                    competitor.threat_level.lower(),
                    "medium"
                ),
                "confidence": 0.65,
                "sources": analysis.sources,
                "verified": False,
            })

        duration = round(
            time.time() - start,
            2
        )

        print(
            f"  [Competitive] "
            f"{len(findings)} findings | "
            f"{duration}s"
        )

        return {
            "competitive_findings": findings,
            "pipeline_trace": [
                {
                    "agent": "competitive",
                    "duration": duration,
                }
            ],
        }

    except Exception as e:

        print(
            f"  [Competitive] FALLBACK: {e}"
        )

        return {
            "competitive_findings": [
                {
                    "agent": "competitive",
                    "category": "competitive",
                    "title": "Competitive analysis failed",
                    "detail": str(e),
                    "severity": "low",
                    "confidence": 0.0,
                    "sources": [],
                    "verified": False,
                }
            ],
            "errors": [str(e)],
        }


print("Competitive Analyst ready.")


Competitive Analyst ready.


In [13]:
# ============================================================
# CELL 12 — RISK ASSESSOR
# ============================================================


def risk_assessor(
    state: DueDiligenceState
) -> dict:

    company = state["company_name"]
    config = get_depth_config(state)

    start = time.time()

    print(
        f"  [Risk] Researching {company}..."
    )

    try:

        tasks = get_agent_tasks(
            state,
            "risk"
        )

        queries = [
            f"{company} lawsuits legal risks",
            f"{company} regulatory risks problems",
            f"{company} operational reputational ESG risks",
            f"{company} technology strategic risks",
        ]

        if tasks:
            queries.append(
                f"{company} " + " ".join(tasks[:2])
            )

        results = []

        for query in queries:

            results.extend(
                web_search(
                    query,
                    config["general_results"]
                )
            )

        context = build_search_context(
            results,
            config["context_sources"]
        )

        planning_context = get_planning_context(
            state,
            "risk"
        )

        structured = llm.with_structured_output(
            RiskAssessment
        )

        analysis = structured.invoke(
            f"""
You are a Risk Assessor conducting company
due diligence.

Company:
{company}

{planning_context}

Identify evidence-supported risks across:
- legal
- regulatory
- operational
- reputational
- financial
- strategic
- technology

For each risk provide severity and likelihood.

Do not invent lawsuits, regulatory actions,
or other facts.

Search results:

{context}
"""
        )

        findings = []

        overall_severity = {
            "critical": "critical",
            "high": "high",
            "moderate": "medium",
            "low": "low",
        }.get(
            analysis.overall_risk_level.lower(),
            "medium"
        )

        findings.append({
            "agent": "risk",
            "category": "risk",
            "title": (
                f"Overall risk: "
                f"{analysis.overall_risk_level}"
            ),
            "detail": analysis.risk_summary,
            "severity": overall_severity,
            "confidence": 0.70,
            "sources": (
                analysis.sources
                or safe_sources(results[:5])
            ),
            "verified": False,
        })

        for risk in analysis.risks:

            severity = risk.severity.lower()

            if severity not in {
                "critical",
                "high",
                "medium",
                "low",
            }:
                severity = "medium"

            sources = (
                [risk.source]
                if risk.source
                else analysis.sources
            )

            findings.append({
                "agent": "risk",
                "category": "risk",
                "title": risk.title,
                "detail": (
                    f"Category: {risk.category}\n"
                    f"Description: {risk.description}\n"
                    f"Likelihood: {risk.likelihood}"
                ),
                "severity": severity,
                "confidence": 0.70,
                "sources": sources,
                "verified": False,
            })

        duration = round(
            time.time() - start,
            2
        )

        print(
            f"  [Risk] "
            f"{analysis.overall_risk_level} | "
            f"{len(analysis.risks)} risks | "
            f"{duration}s"
        )

        return {
            "risk_findings": findings,
            "pipeline_trace": [
                {
                    "agent": "risk",
                    "duration": duration,
                }
            ],
        }

    except Exception as e:

        print(
            f"  [Risk] FALLBACK: {e}"
        )

        return {
            "risk_findings": [
                {
                    "agent": "risk",
                    "category": "risk",
                    "title": "Risk analysis failed",
                    "detail": str(e),
                    "severity": "low",
                    "confidence": 0.0,
                    "sources": [],
                    "verified": False,
                }
            ],
            "errors": [str(e)],
        }


print("Risk Assessor ready.")


Risk Assessor ready.


In [14]:
# ============================================================
# CELL 13 — FACT CHECKER
# ============================================================


def fact_checker(
    state: DueDiligenceState
) -> dict:

    company = state["company_name"]
    config = get_depth_config(state)

    start = time.time()

    print(
        f"  [FactCheck] Verifying findings..."
    )

    try:

        all_findings = (
            state.get("financial_findings", [])
            + state.get("news_findings", [])
            + state.get("competitive_findings", [])
            + state.get("risk_findings", [])
        )

        # Prioritize more serious findings
        priority_order = {
            "critical": 0,
            "high": 1,
            "medium": 2,
            "low": 3,
            "info": 4,
        }

        important_claims = [
            finding
            for finding in all_findings
            if finding.get("severity")
            in {
                "critical",
                "high",
                "medium",
            }
        ]

        important_claims.sort(
            key=lambda x: priority_order.get(
                x.get("severity", "info"),
                99
            )
        )

        important_claims = important_claims[
            :config["fact_check_claims"]
        ]

        if not important_claims:

            print(
                "  [FactCheck] No high-priority "
                "claims to verify."
            )

            return {
                "fact_check_results": [
                    {
                        "total_checked": 0,
                        "verified": 0,
                        "contradicted": 0,
                        "unverifiable": 0,
                        "overall_reliability": "no_claims",
                    }
                ]
            }

        verification_text = ""

        for i, finding in enumerate(
            important_claims,
            start=1
        ):

            title = finding.get(
                "title",
                ""
            )

            detail = finding.get(
                "detail",
                ""
            )

            agent = finding.get(
                "agent",
                "unknown"
            )

            search_query = (
                f"{company} {title}"
            )

            results = web_search(
                search_query,
                3
            )

            evidence = "\n".join(
                [
                    (
                        f"- {r['title']}: "
                        f"{r['snippet'][:300]} "
                        f"({r['url']})"
                    )
                    for r in results
                ]
            )

            verification_text += f"""

CLAIM {i}

Agent:
{agent}

Claim:
{title}

Details:
{detail[:700]}

Independent evidence:
{evidence or "No independent evidence found."}

--------------------------------------------------
"""

        structured = llm.with_structured_output(
            FactCheckReport
        )

        report = structured.invoke(
            f"""
You are an independent fact checker.

Company:
{company}

Review the claims below.

For every claim classify it as:

- verified: independent evidence supports it
- contradicted: independent evidence conflicts with it
- unverifiable: insufficient evidence

Do not assume the original agent is correct.

Also identify meaningful cross-agent contradictions.

Research evidence:

{verification_text}
"""
        )

        contradictions = [
            {
                "claim": contradiction,
                "status": "unresolved",
            }
            for contradiction
            in report.cross_agent_contradictions
        ]

        duration = round(
            time.time() - start,
            2
        )

        print(
            f"  [FactCheck] "
            f"{report.verified_count}/"
            f"{report.total_claims_checked} verified | "
            f"{report.contradicted_count} contradicted | "
            f"{duration}s"
        )

        return {
            "fact_check_results": [
                {
                    "total_checked":
                        report.total_claims_checked,
                    "verified":
                        report.verified_count,
                    "contradicted":
                        report.contradicted_count,
                    "unverifiable":
                        report.unverifiable_count,
                    "overall_reliability":
                        report.overall_reliability,
                    "verifications": [
                        v.model_dump()
                        for v in report.verifications
                    ],
                }
            ],

            "contradictions": contradictions,

            "pipeline_trace": [
                {
                    "agent": "fact_checker",
                    "duration": duration,
                }
            ],
        }

    except Exception as e:

        print(
            f"  [FactCheck] FALLBACK: {e}"
        )

        return {
            "fact_check_results": [
                {
                    "error": str(e),
                    "overall_reliability": "error",
                }
            ],
            "errors": [str(e)],
            "pipeline_trace": [
                {
                    "agent": "fact_checker",
                    "duration": round(
                        time.time() - start,
                        2
                    ),
                }
            ],
        }


print("Fact Checker ready.")


Fact Checker ready.


In [15]:
# ============================================================
# CELL 14 — LEAD ANALYST: SYNTHESIZER
# ============================================================


def synthesize_report(
    state: DueDiligenceState
) -> dict:

    company = state["company_name"]

    start = time.time()

    print(
        f"\n[Lead Analyst] "
        f"Synthesizing report for {company}..."
    )

    try:

        all_findings = []

        for key in [
            "financial_findings",
            "news_findings",
            "competitive_findings",
            "risk_findings",
        ]:

            all_findings.extend(
                state.get(key, [])
            )

        findings_text = ""

        for finding in all_findings:

            findings_text += (
                f"\n"
                f"[{finding.get('severity', '?').upper()}] "
                f"{finding.get('agent', '?')} — "
                f"{finding.get('title', '')}\n"
                f"{finding.get('detail', '')[:700]}\n"
                f"Sources: "
                f"{', '.join(finding.get('sources', []))}\n"
            )

        fact_checks = state.get(
            "fact_check_results",
            []
        )

        fact_check_text = str(
            fact_checks[0]
            if fact_checks
            else "No fact-check information available."
        )

        contradictions = state.get(
            "contradictions",
            []
        )

        contradiction_text = str(
            contradictions
            if contradictions
            else "No cross-agent contradictions detected."
        )

        structured = llm.with_structured_output(
            ExecutiveSummary
        )

        summary = structured.invoke(
            f"""
You are the Lead Analyst responsible for producing
the final executive-level due-diligence assessment.

Company:
{company}

Research objective:
{state.get("query", "")}

Findings:
{findings_text}

Fact-check results:
{fact_check_text}

Detected contradictions:
{contradiction_text}

Produce a balanced assessment.

Important rules:

1. Do not invent facts.
2. Distinguish evidence from uncertainty.
3. Give greater weight to independently verified findings.
4. Mention important unverifiable or contradicted claims.
5. The confidence score should reflect evidence quality,
   source coverage, and fact-checking.
6. Do not claim that contradictions were resolved by debate;
   this version only detects them.

Provide:
- one-line verdict
- overall risk rating
- confidence
- strengths
- risks
- recommendation
- action items
"""
        )

        report = []

        report.append(
            f"# Due Diligence Report: {company}\n"
        )

        report.append(
            f"**Verdict:** "
            f"{summary.one_line_verdict}\n"
        )

        report.append(
            f"**Risk:** "
            f"{summary.overall_risk_rating.upper()}  \n"
            f"**Confidence:** "
            f"{summary.overall_confidence:.0%}  \n"
            f"**Recommendation:** "
            f"{summary.recommendation}\n"
        )

        report.append(
            "## Key Strengths\n"
        )

        report.extend(
            [
                f"- {strength}\n"
                for strength
                in summary.key_strengths
            ]
        )

        report.append(
            "\n## Key Risks\n"
        )

        report.extend(
            [
                f"- {risk}\n"
                for risk
                in summary.key_risks
            ]
        )

        if summary.action_items:

            report.append(
                "\n## Recommended Next Steps\n"
            )

            report.extend(
                [
                    f"{i}. {action}\n"
                    for i, action
                    in enumerate(
                        summary.action_items,
                        start=1
                    )
                ]
            )

        report.append(
            "\n## Fact-Check Summary\n"
        )

        if fact_checks:

            fc = fact_checks[0]

            report.append(
                f"- Claims checked: "
                f"{fc.get('total_checked', 0)}\n"
            )

            report.append(
                f"- Verified: "
                f"{fc.get('verified', 0)}\n"
            )

            report.append(
                f"- Contradicted: "
                f"{fc.get('contradicted', 0)}\n"
            )

            report.append(
                f"- Unverifiable: "
                f"{fc.get('unverifiable', 0)}\n"
            )

            report.append(
                f"- Reliability: "
                f"{fc.get('overall_reliability', 'unknown')}\n"
            )

        else:

            report.append(
                "No fact-check information available.\n"
            )

        if contradictions:

            report.append(
                "\n## Detected Contradictions\n"
            )

            for contradiction in contradictions:

                report.append(
                    f"- {contradiction.get('claim', '')}\n"
                )

        final_report = "\n".join(report)

        duration = round(
            time.time() - start,
            2
        )

        print(
            f"  Verdict: "
            f"{summary.one_line_verdict[:100]}"
        )

        print(
            f"  Risk: "
            f"{summary.overall_risk_rating}"
        )

        print(
            f"  Confidence: "
            f"{summary.overall_confidence:.0%}"
        )

        print(
            f"  Synthesis time: {duration}s"
        )

        return {
            "executive_summary":
                summary.one_line_verdict,

            "final_report":
                final_report,

            "overall_risk_rating":
                summary.overall_risk_rating,

            "overall_confidence":
                summary.overall_confidence,

            "status": "complete",

            "pipeline_trace": [
                {
                    "agent": "lead_synthesis",
                    "duration": duration,
                }
            ],
        }

    except Exception as e:

        print(
            f"  [Synthesis] FALLBACK: {e}"
        )

        return {
            "final_report":
                f"Report generation failed: {e}",

            "status": "complete",

            "errors": [str(e)],
        }


print("Synthesizer ready.")


Synthesizer ready.


In [16]:
# ============================================================
# CELL 15 — LANGGRAPH ORCHESTRATION
# ============================================================


def fan_out_to_specialists(
    state: DueDiligenceState
):
    """
    Send the same shared state to all four specialists.

    LangGraph executes these branches in parallel.
    """

    return [
        Send(
            "financial_analyst",
            state
        ),

        Send(
            "news_sentiment",
            state
        ),

        Send(
            "competitive_intel",
            state
        ),

        Send(
            "risk_assessor",
            state
        ),
    ]


def route_after_fact_check(
    state: DueDiligenceState
):
    """
    Current V1 detects contradictions but does not
    perform a separate debate/resolution loop.

    All results proceed to synthesis.
    """

    return "synthesize_report"


# ------------------------------------------------------------
# Create graph
# ------------------------------------------------------------

graph = StateGraph(
    DueDiligenceState
)


# ------------------------------------------------------------
# Nodes
# ------------------------------------------------------------

graph.add_node(
    "plan_research",
    plan_research
)

graph.add_node(
    "financial_analyst",
    financial_analyst
)

graph.add_node(
    "news_sentiment",
    news_sentiment
)

graph.add_node(
    "competitive_intel",
    competitive_intel
)

graph.add_node(
    "risk_assessor",
    risk_assessor
)

graph.add_node(
    "fact_checker",
    fact_checker
)

graph.add_node(
    "synthesize_report",
    synthesize_report
)


# ------------------------------------------------------------
# Entry
# ------------------------------------------------------------

graph.set_entry_point(
    "plan_research"
)


# ------------------------------------------------------------
# Planner -> parallel specialists
# ------------------------------------------------------------

graph.add_conditional_edges(
    "plan_research",
    fan_out_to_specialists,
    [
        "financial_analyst",
        "news_sentiment",
        "competitive_intel",
        "risk_assessor",
    ]
)


# ------------------------------------------------------------
# Specialists -> fact checker
# ------------------------------------------------------------

graph.add_edge(
    "financial_analyst",
    "fact_checker"
)

graph.add_edge(
    "news_sentiment",
    "fact_checker"
)

graph.add_edge(
    "competitive_intel",
    "fact_checker"
)

graph.add_edge(
    "risk_assessor",
    "fact_checker"
)


# ------------------------------------------------------------
# Fact checker -> synthesis
# ------------------------------------------------------------

graph.add_conditional_edges(
    "fact_checker",
    route_after_fact_check,
    {
        "synthesize_report":
            "synthesize_report"
    }
)


# ------------------------------------------------------------
# End
# ------------------------------------------------------------

graph.add_edge(
    "synthesize_report",
    END
)


# ------------------------------------------------------------
# Compile
# ------------------------------------------------------------

app = graph.compile()

print("LangGraph compiled successfully.")

print(
    "Nodes:",
    list(app.get_graph().nodes.keys())
)


LangGraph compiled successfully.
Nodes: ['__start__', 'plan_research', 'financial_analyst', 'news_sentiment', 'competitive_intel', 'risk_assessor', 'fact_checker', 'synthesize_report', '__end__']


In [17]:
# ============================================================
# CELL 16 — REUSABLE RUN FUNCTION
# ============================================================


def run_due_diligence(
    company: str,
    query: str = "",
    analysis_depth: str = "standard",
) -> dict:
    """
    Run the complete due-diligence pipeline.

    Parameters
    ----------
    company:
        Company to investigate.

    query:
        Optional research objective.

    analysis_depth:
        quick, standard, or deep.
    """

    if analysis_depth not in SEARCH_CONFIG:
        raise ValueError(
            "analysis_depth must be one of: "
            "quick, standard, deep"
        )

    initial_state: DueDiligenceState = {

        "company_name": company,

        "query": (
            query
            or
            f"Comprehensive due diligence "
            f"analysis of {company}"
        ),

        "analysis_depth": analysis_depth,

        "financial_findings": [],
        "news_findings": [],
        "competitive_findings": [],
        "risk_findings": [],

        "fact_check_results": [],
        "contradictions": [],

        "pipeline_trace": [],
        "errors": [],

        "status": "planning",
    }

    print("=" * 70)
    print(
        f"STARTING DUE DILIGENCE: {company}"
    )
    print(
        f"Analysis depth: {analysis_depth}"
    )
    print("=" * 70)

    start = time.time()

    result = app.invoke(
        initial_state
    )

    duration = round(
        time.time() - start,
        2
    )

    print("\n" + "=" * 70)
    print(
        f"COMPLETE IN {duration}s"
    )
    print(
        f"Risk: "
        f"{result.get('overall_risk_rating', 'unknown')}"
    )
    print(
        f"Confidence: "
        f"{result.get('overall_confidence', 0):.0%}"
    )
    print(
        f"Errors: "
        f"{len(result.get('errors', []))}"
    )
    print("=" * 70)

    return result


print("run_due_diligence() ready.")


run_due_diligence() ready.


In [18]:
# ============================================================
# CELL 17 — RUN ANALYSIS
# ============================================================

COMPANY = "Apple"

QUERY = (
    "Evaluate financial health, competitive position, "
    "recent developments, and major business risks."
)

result = run_due_diligence(
    company=COMPANY,
    query=QUERY,
    analysis_depth="standard",
)


STARTING DUE DILIGENCE: Apple
Analysis depth: standard

[Lead Analyst] Planning research for Apple
  Created 4 tasks in 23.47s
  [Financial] Researching Apple...
  [News] Researching Apple...
  [Competitive] Researching Apple...
  [Risk] Researching Apple...
Search failed for 'Apple recent controversy criticism': No results found.
  [News] 4 events | 23.64s
  [Financial] strong | 8 findings | 24.5s
  [Risk] high | 4 risks | 28.06s
  [Competitive] 14 findings | 29.21s
  [FactCheck] Verifying findings...
Search failed for 'Apple Apple unveils next generation of Apple Intelligence and Siri AI': ConnectError: ConnectError('error sending request for url (https://grokipedia.com/api/typeahead?query=Apple+Apple+unveils+next+generation+of+Apple+Intelligence+and+Siri+AI&limit=1) > Network is unreachable (os error 101)')
  [FactCheck] 6/8 verified | 0 contradicted | 36.59s

[Lead Analyst] Synthesizing report for Apple...
  Verdict: Apple exhibits stellar financial health and market dominance, but

In [19]:
# ============================================================
# CELL 18 — DISPLAY FINAL REPORT
# ============================================================

from IPython.display import Markdown, display

report = result.get(
    "final_report",
    "No report generated."
)

display(
    Markdown(report)
)


# Due Diligence Report: Apple

**Verdict:** Apple exhibits stellar financial health and market dominance, but faces mounting regulatory, legal, and supply chain pressures.

**Risk:** MEDIUM  
**Confidence:** 80%  
**Recommendation:** Maintain a positive outlook on Apple due to its unmatched financial strength and ecosystem lock-in, but factor in potential valuation headwinds from regulatory actions against its high-margin Services division.

## Key Strengths

- Exceptional profitability with an EBITDA of $144.7 billion in FY 2025 and robust free cash flow.

- Consistent double-digit quarterly revenue growth throughout FY 2026.

- Strong liquidity position with $35.9 billion in cash and equivalents as of FY 2025.

- Dominant market position in consumer electronics, including a 32% global market share for iPad.


## Key Risks

- Intense regulatory and antitrust scrutiny targeting App Store monetization and platform control.

- Significant long-term debt of $78.3 billion and $7.98 billion in outstanding commercial paper.

- Relentless competition in the smartphone and tablet markets from Samsung, Google, and Chinese brands.

- Unverifiable but noted risks concerning supply chain concentration in China and potential false advertising litigation over AI features.


## Recommended Next Steps

1. Conduct deeper due diligence on the progress of supply chain diversification away from China.

2. Monitor the legal developments and financial impact of global antitrust lawsuits targeting the App Store.

3. Track the launch, consumer adoption, and legal compliance of Apple Intelligence and rumored hardware releases.

4. Evaluate the sustainability of Apple's capital return program (buybacks/dividends) against its debt structure.


## Fact-Check Summary

- Claims checked: 8

- Verified: 6

- Contradicted: 0

- Unverifiable: 2

- Reliability: moderate


In [20]:
# ============================================================
# CELL 19 — PIPELINE TRACE
# ============================================================

print("PIPELINE EXECUTION")
print("-" * 50)

for trace in result.get(
    "pipeline_trace",
    []
):

    print(
        f"{trace.get('agent', '?'):25s} "
        f"{trace.get('duration', 0):6.2f}s"
    )


print("\nFINDINGS")
print("-" * 50)

print(
    "Financial:",
    len(
        result.get(
            "financial_findings",
            []
        )
    )
)

print(
    "News:",
    len(
        result.get(
            "news_findings",
            []
        )
    )
)

print(
    "Competitive:",
    len(
        result.get(
            "competitive_findings",
            []
        )
    )
)

print(
    "Risk:",
    len(
        result.get(
            "risk_findings",
            []
        )
    )
)


print("\nFACT CHECK")
print("-" * 50)

for fc in result.get(
    "fact_check_results",
    []
):

    print(fc)


print("\nERRORS")
print("-" * 50)

errors = result.get(
    "errors",
    []
)

if errors:

    for error in errors:
        print("-", error)

else:

    print("No errors.")


PIPELINE EXECUTION
--------------------------------------------------
lead_planner               23.47s
financial                  24.50s
news                       23.64s
competitive                29.21s
risk                       28.06s
fact_checker               36.59s
lead_synthesis              9.46s

FINDINGS
--------------------------------------------------
Financial: 8
News: 5
Competitive: 14
Risk: 5

FACT CHECK
--------------------------------------------------
{'total_checked': 8, 'verified': 6, 'contradicted': 0, 'unverifiable': 2, 'overall_reliability': 'moderate', 'verifications': [{'claim': 'Financial concern: Significant long-term debt of $78.3 billion (or $78.33 billion) as of fiscal year 2025.', 'agent': 'financial', 'status': 'verified', 'reasoning': "Multiple independent sources confirm Apple's long-term debt was approximately $78.33 billion (or $78.328 billion) for the fiscal year ending September 2025.", 'supporting_sources': ['https://unanswered.io/guide/how-muc

In [21]:
# ============================================================
# CELL 20 — INSPECT RESEARCH PLAN
# ============================================================

import pprint

print("GENERATED RESEARCH PLAN")
print("=" * 70)

pprint.pp(
    result.get(
        "research_plan",
        {}
    )
)

print("\nFOCUS AREAS")
print("=" * 70)

for area in result.get(
    "focus_areas",
    []
):
    print("-", area)


GENERATED RESEARCH PLAN
{'summary': 'Apple Inc. is a global technology leader specializing in consumer '
            'electronics, software, and services, known for its flagship '
            'iPhone, Mac, iPad, and a rapidly expanding Services division.',
 'sub_tasks': [{'agent': 'financial',
                'task': "Analyze Apple's latest 10-K and 10-Q filings to "
                        'evaluate revenue growth trends across product '
                        'segments (iPhone, Services, Wearables) and assess '
                        'operating margins and free cash flow generation.',
                'priority': 'high'},
               {'agent': 'news',
                'task': 'Search for recent press releases and news articles '
                        "from the past 6 months regarding Apple's AI "
                        'initiatives (Apple Intelligence), product launches, '
                        'and executive leadership changes.',
                'priority': 'medium'},
      

In [22]:
# ============================================================
# CELL 21 — SECOND EXAMPLE
# ============================================================

result2 = run_due_diligence(
    company="Stripe",
    query=(
        "Focus on fintech competition, "
        "regulatory exposure, and business risks."
    ),
    analysis_depth="quick",
)

display(
    Markdown(
        result2.get(
            "final_report",
            "No report generated."
        )
    )
)


STARTING DUE DILIGENCE: Stripe
Analysis depth: quick

[Lead Analyst] Planning research for Stripe
  Created 4 tasks in 8.39s
  [Financial] Researching Stripe...
  [News] Researching Stripe...
  [Competitive] Researching Stripe...
  [Risk] Researching Stripe...
Search failed for 'Stripe latest annual report financial results revenue profit': No results found.
Search failed for 'Stripe industry market position comparison': ConnectError: ConnectError('error sending request for url (https://www.google.com/wml/search?q=Stripe+industry+market+position+comparison&sca_esv=1&filter=1&start=0&hl=en-US&lr=lang_en&cr=countryUS) > Network is unreachable (os error 101)')
  [News] 5 events | 12.52s
  [Competitive] FALLBACK: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/

KeyboardInterrupt: 